**German Credit Dataset**

# 03.1 - Classic Models Pipeline

**Objectives**
- Build and run machine learning pipelines with different configurations
- Train and evaluate several classical models automatically
- Export performance and fairness results for all model configurations


In [ ]:
import os
import time
import sys
import importlib

from itertools import product
import pandas as pd
import random
import numpy as np
from sklearn.model_selection import StratifiedKFold
from aif360.datasets import BinaryLabelDataset

In [ ]:
sys.path.append('utils')

os.environ['PYTHONHASHSEED'] = '2'
random.seed(2)
np.random.seed(2)

In [45]:
import preprocessing
import bias_preprocessing
import models
import tuning
import bias_postprocessing
import evaluation

importlib.reload(preprocessing)
importlib.reload(bias_preprocessing)
importlib.reload(models)
importlib.reload(tuning)
importlib.reload(bias_postprocessing)
importlib.reload(evaluation)


<module 'evaluation' from 'c:\\Users\\dudab\\Projects\\ml-fairness-credit-analysis\\new_new_pipeline\\utils\\evaluation.py'>

## 1. Load Data

In [46]:
file_path = '../data/processed/german_df_processed_2.csv'
df = pd.read_csv(file_path, sep=r',', header=0)

In [47]:
df

,checking_account_status,duration_months,credit_history,purpose,credit_amount,savings_account_status,employment_status,installment_rate,guarantors,residence_duration,...,other_debts,housing,existing_credits_count,job,dependents,own_telephone?,foreign_worker?,good_client?,sex,age_cat
0,1,6,5,4,1169,5,5,4,1,4,...,3,2,2,3,1,1,1,1,1,1
1,2,48,3,4,5951,1,3,2,1,2,...,3,2,1,3,1,0,1,0,2,2
2,4,12,5,7,2096,1,4,2,1,3,...,3,2,1,2,2,0,1,1,1,1
3,1,42,3,3,7882,1,4,2,3,4,...,3,3,1,3,2,0,1,1,1,1
4,1,24,4,1,4870,1,3,3,1,4,...,3,3,2,3,2,0,1,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,4,12,3,3,1736,1,4,3,1,4,...,3,2,1,2,1,0,1,1,2,1
996,1,30,3,2,3857,1,3,4,1,4,...,3,2,1,4,1,1,1,1,1,1
997,4,12,3,4,804,1,5,4,1,4,...,3,2,1,3,1,0,1,1,1,1
998,1,45,3,4,1845,1,3,4,1,4,...,3,3,1,3,1,1,1,0,1,2


In [48]:
df.columns

Index(['checking_account_status', 'duration_months', 'credit_history',
       'purpose', 'credit_amount', 'savings_account_status',
       'employment_status', 'installment_rate', 'guarantors',
       'residence_duration', 'property', 'age', 'other_debts', 'housing',
       'existing_credits_count', 'job', 'dependents', 'own_telephone?',
       'foreign_worker?', 'good_client?', 'sex', 'age_cat'],
      dtype='str')

In [49]:
(df['foreign_worker?'] == 0).sum()

37

## 2. Setup: Key Variables, Helper Functions, and Pipeline Construction

### 2.1. Key Variables

In [50]:
NUMERIC_COLS = [
    'duration_months', 'credit_amount'
]

NOMINAL_COLS = [
    'sex', 'age_cat', 'checking_account_status', 'credit_history',
    'purpose', 'savings_account_status', 'employment_status',
    'guarantors', 'property', 'other_debts', 'housing', 'job',
    'own_telephone?', 'foreign_worker?','dependents'
]

ORDINAL_COLS = [
    'installment_rate', 'residence_duration', 'existing_credits_count'
]

CATEGORICAL_COLS = NOMINAL_COLS + ORDINAL_COLS

In [ ]:
TARGET_COLUMN = 'good_client?'
FAVORABLE_LABEL = 1   # 'Good client'
UNFAVORABLE_LABEL = 0 # 'No good client'

SENSITIVE_ATTR = 'sex'
PRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 1}]   # Male
UNPRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 2}] # Female

# SENSITIVE_ATTR = 'foreign_worker?'
# PRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 1}]   # Foreign
# UNPRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 0}] # Local

# SENSITIVE_ATTR = 'age_cat'
# PRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 1}]   # age >= 25
# UNPRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 2}] # age < 25 

In [ ]:
# 1) Pre-processing
SCALERS = ['none', 'standardization']
ENCODERS = ['none', 'one-hot', 'label']

# 2) Pre-processing bias mitigation
BIAS_PRE = ['none', 'reweighing' ,'disparate-impact-remover']

# 3) Models
MODELS = ['logistic_regression', 'random_forest', 'gradient_boosting']

# 4) Tuning 
TUNING = ['none', 'random']

# 5) Post-processing bias mitigation
BIAS_POST = ['none', 'calibrate_equalized_odds', 'reject_option_classification']


### 2.2. Helper Functions

In [53]:
def df_to_aif360(df):
    return BinaryLabelDataset(
        df=df,
        label_names=[TARGET_COLUMN],
        protected_attribute_names=[SENSITIVE_ATTR],
        favorable_label=FAVORABLE_LABEL,
        unfavorable_label=UNFAVORABLE_LABEL
    )

### 2.3. Pipeline Construction

In [54]:
def run_pipeline(config, df_train, df_test, verbose_diag=False):
    """
    Executes a single pipeline configuration on a fixed train/test split.

      1) Separates X/y from the raw dataframes
      2) Applies standard pre-processing using sklearn (scaling/encoding)
      3) Converts the alredy pre-processed data to BinaryLabelDataset (AIF360)
      4) Applies pre-processing bias mitigation
      5) Trains the model (with or without tuning), using sample_weight if available
      6) Generates scores (probabilities) on the test set
      7) Applies post-processing bias mitigation
      8) Evaluates performance and fairness, and returns the metrics

    Args:
        config (dict): Pipeline configuration dict.
        df_train (pd.DataFrame): Training data.
        df_test (pd.DataFrame): Test/validation data.
        verbose_diag (bool): If True, print logs.

    Returns:
        pd.DataFrame: Metrics DataFrame (one row).
    """
    print(f"Running: {config['id']}")

    # ------------------------------------------------------------------
    #  1) Separates X/y from the raw dataframes
    # ------------------------------------------------------------------
    feature_cols = list(NUMERIC_COLS) + list(CATEGORICAL_COLS)

    X_train_df = df_train[feature_cols].copy()
    X_test_df  = df_test[feature_cols].copy()
    y_train    = df_train[TARGET_COLUMN].values
    y_test     = df_test[TARGET_COLUMN].values

    # To keep a safe copy of the sentitive atribute
    prot_train_raw = df_train[SENSITIVE_ATTR].values
    prot_test_raw  = df_test[SENSITIVE_ATTR].values

    # ------------------------------------------------------------------
    #  2) Applies standard pre-processing (scaling/encoding) 
    # ------------------------------------------------------------------
    preprocessor = preprocessing.build_preprocessor(
        NUMERIC_COLS, CATEGORICAL_COLS, config['scaler'], config['encoder']
    )
    X_train_arr = preprocessor.fit_transform(X_train_df)
    X_test_arr  = preprocessor.transform(X_test_df)

    feat_names = list(preprocessor.get_feature_names_out())
    X_train_proc_df = pd.DataFrame(X_train_arr, columns=feat_names, index=X_train_df.index)
    X_test_proc_df  = pd.DataFrame(X_test_arr,  columns=feat_names, index=X_test_df.index)

    # ------------------------------------------------------------------
    #  3) Converts the alredy pre-processed data to BinaryLabelDataset
    # ------------------------------------------------------------------
    ds_train_df = X_train_proc_df.copy()
    ds_train_df[SENSITIVE_ATTR] = prot_train_raw
    ds_train_df[TARGET_COLUMN]  = y_train

    ds_test_df = X_test_proc_df.copy()
    ds_test_df[SENSITIVE_ATTR] = prot_test_raw
    ds_test_df[TARGET_COLUMN]  = y_test

    ds_train = df_to_aif360(ds_train_df)
    ds_test  = df_to_aif360(ds_test_df)

    # ------------------------------------------------------------------
    #  4) Applies pre-processing bias mitigation
    # ------------------------------------------------------------------

    repairable_raw = list(NUMERIC_COLS) + list(ORDINAL_COLS)
    features_to_repair = [f for f in feat_names if f in repairable_raw]
    if config['bias_pre'] == 'disparate-impact-remover' and len(features_to_repair) < len(repairable_raw):
        print(
            f"[AVISO] encoder='{config['encoder']}' expandiu/renomeou algumas colunas "
            f"de NUMERIC_COLS+ORDINAL_COLS. DIR vai reparar {len(features_to_repair)} de "
            f"{len(repairable_raw)} colunas esperadas: {features_to_repair}"
        )
    ds_train_bias_proc = bias_preprocessing.apply_bias_preprocessing(
        config['bias_pre'], ds_train, SENSITIVE_ATTR, UNPRIVILEGED_GROUPS, PRIVILEGED_GROUPS,
        features_to_repair=features_to_repair,
    )

    X_train_final = pd.DataFrame(
        ds_train_bias_proc.features,
        columns=ds_train_bias_proc.feature_names
    )
    X_test_final = pd.DataFrame(
        ds_test.features,
        columns=ds_test.feature_names
    )

    # Remove the raw sensitive attribute. 
    # Its encoded version is already included 
    X_train_final = X_train_final.drop(columns=[SENSITIVE_ATTR], errors='ignore')
    X_test_final  = X_test_final.drop(columns=[SENSITIVE_ATTR], errors='ignore')

    y_train_final = ds_train_bias_proc.labels.ravel()
    sample_weight = ds_train_bias_proc.instance_weights.ravel()

    # ------------------------------------------------------------------
    #  5) Trains the model using sample_weight if available
    # ------------------------------------------------------------------
    base_model = models.get_model(config['model'])
    grid_p, random_p = models.get_hyperparameters(config['model'])
    model = tuning.apply_tuning(config['tuning'], base_model, grid_p, random_p)

    model.fit(X_train_final, y_train_final, sample_weight=sample_weight)

    # ------------------------------------------------------------------
    #  6) Generates scores (probabilities) on the test set
    # ------------------------------------------------------------------
    ds_pred = ds_test.copy()
    ds_pred.scores = model.predict_proba(X_test_final)[:, 1].reshape(-1, 1)


    # ------------------------------------------------------------------
    #  7) Applies post-processing bias mitigation
    # ------------------------------------------------------------------
    ds_pred_mitigated = bias_postprocessing.apply_bias_postprocessing(
        config['bias_post'], ds_test, ds_pred, UNPRIVILEGED_GROUPS, PRIVILEGED_GROUPS
    )

    # ------------------------------------------------------------------
    #  8) Evaluates performance and fairness, and returns the metrics
    # ------------------------------------------------------------------
    return evaluation.evaluate_pipeline(
        ds_test, ds_pred_mitigated,
        UNPRIVILEGED_GROUPS, PRIVILEGED_GROUPS,
        config['id']
    )


In [55]:
def run_cv_pipeline(
    config, df,
    n_splits=4,
    random_state=2,
    shuffle=True,
    verbose_diag=False
):
    """
    Executes Cross-Validation (StratifiedKFold) for a single pipeline configuration.

    Args:
        config (dict): Pipeline configuration.
        df (pd.DataFrame): Complete dataset.
        n_splits (int): Number of folds.
        random_state (int)
        shuffle (bool)
        verbose_diag (bool)

    Returns:
        pd.DataFrame: Aggregated metrics and model configutration (one row).
    """

    skf = StratifiedKFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
    y = df[TARGET_COLUMN].values

    fold_rows = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(df, y), start=1):
        df_train_fold = df.iloc[train_idx].copy()
        df_test_fold  = df.iloc[test_idx].copy()

        df_fold = run_pipeline(
            config,
            df_train=df_train_fold,
            df_test=df_test_fold,
            verbose_diag=verbose_diag
        )

        row = df_fold.iloc[0].to_dict()
        row.pop('Pipeline', None)
        fold_rows.append(row)

    df_tmp = pd.DataFrame(fold_rows)

    means = df_tmp.mean(numeric_only=True)
    stds  = df_tmp.std(numeric_only=True)

    summary_row = {
        'config_id':  config['id'],
        'model':      config.get('model'),
        'source':     config.get('source', 'classic'),
        'scaler':     config.get('scaler'),
        'encoder':    config.get('encoder'),
        'bias_pre':   config.get('bias_pre'),
        'tuning':     config.get('tuning'),
        'bias_post':  config.get('bias_post'),
        'n_splits':   n_splits,
    }
    for col, val in means.items():
        summary_row[f'mean_{col}'] = float(val)
    for col, val in stds.items():
        summary_row[f'std_{col}'] = float(val)

    return pd.DataFrame([summary_row])


In [ ]:
MODEL_PREFIX = {
    "logistic_regression": "LG",
    "random_forest": "RF",
    "gradient_boosting": "GB"
}


def _assign_model_id(results):
    prefix = results['model'].map(MODEL_PREFIX).fillna('MD')
    rank = prefix.groupby(prefix).cumcount() + 1
    results = results.copy()
    results['model_id'] = prefix + rank.astype(str).str.zfill(3)
    return results


def _reorder_results_columns(results):
    front = [
        'model_id', 'model', 'source', 'scaler', 'encoder', 'bias_pre',
        'tuning', 'bias_post', 'n_splits',
        'eta', 'adversary_loss_weight', 'num_epochs',
        'classifier_num_hidden_units', 'debias',
    ]
    front = [c for c in front if c in results.columns]
    metrics = [c for c in results.columns if c.startswith('mean_') or c.startswith('std_')]
    rest = [c for c in results.columns if c not in front + metrics]
    return results[front + metrics + rest]

In [57]:
def run_all_combinations(
    df,
    n_splits=4,
    random_state=2,
    shuffle=True,
    fail_fast=False
):
    """
    Runs ALL possible pipeline combinations with cross-validation.

    Args:
        df (pd.DataFrame).
        n_splits (int).
        random_state (int).
        shuffle (bool).
        fail_fast (bool): If True, stops at the first error.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: (results_cv_df, errors_df).
    """

    # ------------------------------------------------------------------
    # 1) Generate all pipeline configuration combinations
    # ------------------------------------------------------------------
    configs = []
    for scaler, encoder, bias_pre, model, tuning_method, bias_post in product(
        SCALERS, ENCODERS, BIAS_PRE, MODELS, TUNING, BIAS_POST
    ):
        configs.append({
            'id': f'{model}|{scaler}|{encoder}|pre={bias_pre}|tune={tuning_method}|post={bias_post}',
            'scaler':    scaler,
            'encoder':   encoder,
            'bias_pre':  bias_pre,
            'model':     model,
            'tuning':    tuning_method,
            'bias_post': bias_post,
            'source':    'classic',
        })

    total = len(configs)
    print(f"Total combinations: {total} | n_splits={n_splits} | shuffle={shuffle} | random_state={random_state}")

    summaries = []
    errors    = []
    start_all = time.time()

    # ------------------------------------------------------------------
    # 2) Run CV for each configuration
    # ------------------------------------------------------------------
    for i, cfg in enumerate(configs, start=1):
        print(f"\n[{i}/{total}] Running: {cfg['id']}")

        try:
            df_summary = run_cv_pipeline(
                config=cfg,
                df=df,
                n_splits=n_splits,
                random_state=random_state,
                shuffle=shuffle
            )
            summaries.append(df_summary)

        except Exception as e:
            print(f"[ERROR] {cfg['id']}: {e}")
            errors.append({'config_id': cfg['id'], 'error': str(e)})
            if fail_fast:
                raise

    elapsed = time.time() - start_all
    print(f"\nConcluído em {elapsed:.1f}s | sucesso={len(summaries)} | erros={len(errors)}")

    if summaries:
        results = pd.concat(summaries, ignore_index=True)
        results = _assign_model_id(results)
        results = _reorder_results_columns(results)
    else:
        results = pd.DataFrame()

    errors_df = pd.DataFrame(errors)

    return results, errors_df


## 3. Pipeline Execution

### 3.1. Single model test

In [58]:
df_summary = run_cv_pipeline(
    config={
        'id':       'German Credit Single Model Test',
        'scaler':   'standardization',
        'encoder':  'none',
        'bias_pre': 'disparate-impact-remover',
        'model':    'logistic_regression',
        'tuning':   'random',
        'bias_post':'none',
    },
    df=df,
    n_splits=3,
    verbose_diag=True   
)


Running: German Credit Single Model Test
Fitting 5 folds for each of 25 candidates, totalling 125 fits

 RESULTS: GERMAN CREDIT SINGLE MODEL TEST

--- Performance Metrics ---
Accuracy:            0.7305
Precision:           0.8636
Recall:              0.7308
F1-Score:            0.7917

--- Fairness Metrics (Ideally close to 0.0) ---
Demographic Parity Diff.:          -0.0800
Equal Opportunity Diff.:           +0.0034
Predictive Parity Diff.:           -0.0237
Average Predictive Value Diff.:    +0.0686
Average Odds Diff.:                -0.0410

Running: German Credit Single Model Test
Fitting 5 folds for each of 25 candidates, totalling 125 fits

 RESULTS: GERMAN CREDIT SINGLE MODEL TEST

--- Performance Metrics ---
Accuracy:            0.7447
Precision:           0.7868
Recall:              0.8712
F1-Score:            0.8269

--- Fairness Metrics (Ideally close to 0.0) ---
Demographic Parity Diff.:          -0.0750
Equal Opportunity Diff.:           -0.0477
Predictive Parity Diff.:  

In [59]:
df_summary

,config_id,model,source,scaler,encoder,bias_pre,tuning,bias_post,n_splits,mean_Accuracy,...,mean_Average Odds Diff.,std_Accuracy,std_Precision,std_Recall,std_F1-Score,std_Demographic Parity Diff.,std_Equal Opportunity Diff.,std_Predictive Parity Diff.,std_Average Predictive Value Diff.,std_Average Odds Diff.
0,German Credit Single Model Test,logistic_regression,classic,standardization,none,disparate-impact-remover,random,none,3,0.75102,...,-0.017777,0.024237,0.041077,0.094254,0.030029,0.065295,0.054416,0.041057,0.054894,0.040354


### 3.2. All models

In [60]:
results, errors = run_all_combinations(df=df, n_splits=4)

Total combinations: 108 | n_splits=4 | shuffle=True | random_state=2

[1/108] Running: logistic_regression|none|none|pre=disparate-impact-remover|tune=none|post=none
Running: logistic_regression|none|none|pre=disparate-impact-remover|tune=none|post=none
Tuning disabled. Using the model default parameters.

 RESULTS: LOGISTIC_REGRESSION|NONE|NONE|PRE=DISPARATE-IMPACT-REMOVER|TUNE=NONE|POST=NONE

--- Performance Metrics ---
Accuracy:            0.7360
Precision:           0.7711
Recall:              0.8857
F1-Score:            0.8245

--- Fairness Metrics (Ideally close to 0.0) ---
Demographic Parity Diff.:          -0.0243
Equal Opportunity Diff.:           +0.0229
Predictive Parity Diff.:           -0.0597
Average Predictive Value Diff.:    +0.0575
Average Odds Diff.:                -0.0106

Running: logistic_regression|none|none|pre=disparate-impact-remover|tune=none|post=none
Tuning disabled. Using the model default parameters.

 RESULTS: LOGISTIC_REGRESSION|NONE|NONE|PRE=DISPARATE-I

In [61]:
results.sort_values('mean_F1-Score', ascending=False).head(20)

,model_id,model,source,scaler,encoder,bias_pre,tuning,bias_post,n_splits,mean_Accuracy,...,std_Accuracy,std_Precision,std_Recall,std_F1-Score,std_Demographic Parity Diff.,std_Equal Opportunity Diff.,std_Predictive Parity Diff.,std_Average Predictive Value Diff.,std_Average Odds Diff.,config_id
48,GB013,gradient_boosting,classic,none,label,disparate-impact-remover,none,none,4,0.782,...,0.012000,0.009827,0.028524,0.010610,0.074560,0.054794,0.071365,0.089625,0.075703,gradient_boosting|none|label|pre=disparate-imp...
12,GB001,gradient_boosting,classic,none,none,disparate-impact-remover,none,none,4,0.782,...,0.012000,0.009827,0.028524,0.010610,0.074560,0.054794,0.071365,0.089625,0.075703,gradient_boosting|none|none|pre=disparate-impa...
102,GB031,gradient_boosting,classic,standardization,label,disparate-impact-remover,none,none,4,0.782,...,0.012000,0.009827,0.028524,0.010610,0.074560,0.054794,0.071365,0.089625,0.075703,gradient_boosting|standardization|label|pre=di...
66,GB019,gradient_boosting,classic,standardization,none,disparate-impact-remover,none,none,4,0.782,...,0.012000,0.009827,0.028524,0.010610,0.074560,0.054794,0.071365,0.089625,0.075703,gradient_boosting|standardization|none|pre=dis...
105,GB034,gradient_boosting,classic,standardization,label,disparate-impact-remover,random,none,4,0.782,...,0.022030,0.005002,0.038190,0.018423,0.052821,0.054733,0.071592,0.079486,0.022255,gradient_boosting|standardization|label|pre=di...
51,GB016,gradient_boosting,classic,none,label,disparate-impact-remover,random,none,4,0.782,...,0.022030,0.005002,0.038190,0.018423,0.052821,0.054733,0.071592,0.079486,0.022255,gradient_boosting|none|label|pre=disparate-imp...
15,GB004,gradient_boosting,classic,none,none,disparate-impact-remover,random,none,4,0.781,...,0.022716,0.005498,0.038580,0.018851,0.053090,0.052738,0.071306,0.076544,0.024158,gradient_boosting|none|none|pre=disparate-impa...
69,GB022,gradient_boosting,classic,standardization,none,disparate-impact-remover,random,none,4,0.781,...,0.022716,0.006100,0.037471,0.018689,0.050912,0.048217,0.067010,0.068003,0.024157,gradient_boosting|standardization|none|pre=dis...
106,GB035,gradient_boosting,classic,standardization,label,disparate-impact-remover,random,calibrate_equalized_odds,4,0.766,...,0.020000,0.022875,0.034444,0.013884,0.113050,0.075464,0.127993,0.060517,0.142567,gradient_boosting|standardization|label|pre=di...
52,GB017,gradient_boosting,classic,none,label,disparate-impact-remover,random,calibrate_equalized_odds,4,0.766,...,0.020000,0.022875,0.034444,0.013884,0.113050,0.075464,0.127993,0.060517,0.142567,gradient_boosting|none|label|pre=disparate-imp...


In [62]:
errors

""


## 4. Exporting Results

In [63]:
file_out_path = '../data/results'
results.to_csv(file_out_path + '/classic_models_results_sex_DIR.csv', index=False)